In [32]:
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    classification_report
)

from Bio import pairwise2

warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================
METADATA_FILE = "testset_A_Ak_CAL_table.xlsx"

OUTPUT_RESULTS_XLSX = "testset_sequence_identity_baseline_5fold_results.xlsx"
OUTPUT_RESULTS_TSV = "testset_sequence_identity_baseline_5fold_results.tsv"
OUTPUT_PRED_XLSX = "testset_sequence_identity_baseline_fold_predictions.xlsx"
OUTPUT_PER_CLASS_XLSX = "testset_sequence_identity_baseline_per_class_results.xlsx"
OUTPUT_LABEL_MAP_XLSX = "testset_sequence_identity_baseline_label_mapping.xlsx"

RANDOM_STATE = 42
N_SPLITS = 5

TARGET_CLASSES = ["A_normal", "Ak", "CAL"]


# =========================================================
# 1. LOAD DATA
# =========================================================
if METADATA_FILE.endswith(".xlsx"):
    df = pd.read_excel(METADATA_FILE)
elif METADATA_FILE.endswith(".csv"):
    df = pd.read_csv(METADATA_FILE)
elif METADATA_FILE.endswith(".tsv"):
    df = pd.read_csv(METADATA_FILE, sep="\t")
else:
    raise ValueError("Unsupported metadata file format")

print("=" * 60)
print("Loaded metadata file:", METADATA_FILE)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
print("=" * 60)


# =========================================================
# 2. CHECK REQUIRED COLUMNS
# =========================================================
required_cols = ["sequence"]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if "domain_label" in df.columns:
    label_col = "domain_label"
elif "domain_type" in df.columns:
    label_col = "domain_type"
else:
    raise ValueError("Could not find 'domain_label' or 'domain_type' column.")

if "id" not in df.columns:
    df["id"] = [f"seq_{i}" for i in range(len(df))]


# =========================================================
# 3. CLEAN LABELS AND SEQUENCES
# =========================================================
def normalize_label(x):
    s = str(x).strip().upper().replace("-", "_").replace(" ", "_")

    if s in {"A", "A_NORMAL", "ANORMAL", "A_NORMAL_DOMAIN"}:
        return "A_normal"

    if s in {"AK", "AK_DOMAIN", "AMP", "AMP_BINDING", "AMPBINDING"}:
        return "Ak"

    if s in {"CAL", "CAL_DOMAIN", "CALDOMAIN"}:
        return "CAL"

    return x


df["domain_label"] = df[label_col].apply(normalize_label)

df["sequence"] = (
    df["sequence"]
    .astype(str)
    .str.upper()
    .str.replace(r"[^A-Z]", "", regex=True)
)

df = df[df["domain_label"].isin(TARGET_CLASSES)].copy()
df = df[df["sequence"] != ""].copy().reset_index(drop=True)

print("\nFinal label counts:")
print(df["domain_label"].value_counts())

if df["domain_label"].nunique() != 3:
    raise ValueError(f"Expected 3 classes, found: {df['domain_label'].unique()}")


# =========================================================
# 4. OPTIONAL: REMOVE EXACT DUPLICATES
# =========================================================
before = len(df)
df = df.drop_duplicates(subset=["sequence", "domain_label"]).reset_index(drop=True)
after = len(df)
print(f"\nRemoved exact duplicate sequence+label rows: {before - after}")
print("Rows after deduplication:", len(df))


# =========================================================
# 5. ENCODE LABELS
# =========================================================
label_order = ["A_normal", "Ak", "CAL"]
label_to_int = {label: i for i, label in enumerate(label_order)}
int_to_label = {i: label for label, i in label_to_int.items()}

y = df["domain_label"].map(label_to_int).values
seqs = df["sequence"].tolist()
ids = df["id"].tolist()

print("\nEncoded classes:")
for label in label_order:
    print(label_to_int[label], "->", label)


# =========================================================
# 6. SEQUENCE IDENTITY FUNCTION
# =========================================================
def calc_sequence_identity(seq1, seq2):
    """
    Global sequence identity baseline.
    Identity = matches / alignment length
    """
    aln = pairwise2.align.globalxx(seq1, seq2, one_alignment_only=True)[0]
    aligned_seq1 = aln.seqA
    aligned_seq2 = aln.seqB

    matches = sum(a == b for a, b in zip(aligned_seq1, aligned_seq2))
    aln_len = len(aligned_seq1)

    if aln_len == 0:
        return 0.0

    return matches / aln_len


# =========================================================
# 7. CROSS-VALIDATION
# =========================================================
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

accs, f1s, precs, recs = [], [], [], []
all_fold_predictions = []
per_class_results = []

for fold, (train_idx, test_idx) in enumerate(cv.split(seqs, y), start=1):
    print("\n" + "=" * 60)
    print(f"Fold {fold}")
    print("=" * 60)

    train_seqs = [seqs[i] for i in train_idx]
    train_labels = [y[i] for i in train_idx]
    train_ids = [ids[i] for i in train_idx]

    test_seqs = [seqs[i] for i in test_idx]
    test_labels = [y[i] for i in test_idx]
    test_ids = [ids[i] for i in test_idx]

    y_pred = []
    best_identities = []
    best_train_ids = []

    for j, test_seq in enumerate(test_seqs):
        best_identity = -1.0
        best_label = None
        best_train_id = None

        for tr_seq, tr_label, tr_id in zip(train_seqs, train_labels, train_ids):
            ident = calc_sequence_identity(test_seq, tr_seq)

            if ident > best_identity:
                best_identity = ident
                best_label = tr_label
                best_train_id = tr_id

        y_pred.append(best_label)
        best_identities.append(best_identity)
        best_train_ids.append(best_train_id)

        if (j + 1) % 20 == 0 or (j + 1) == len(test_seqs):
            print(f"  Processed {j+1}/{len(test_seqs)} test sequences")

    acc = accuracy_score(test_labels, y_pred)
    f1 = f1_score(test_labels, y_pred, average="macro", zero_division=0)
    prec = precision_score(test_labels, y_pred, average="macro", zero_division=0)
    rec = recall_score(test_labels, y_pred, average="macro", zero_division=0)

    accs.append(acc)
    f1s.append(f1)
    precs.append(prec)
    recs.append(rec)

    print(f"\nFold {fold} metrics:")
    print(f"  Accuracy        = {acc:.4f}")
    print(f"  F1 macro        = {f1:.4f}")
    print(f"  Precision macro = {prec:.4f}")
    print(f"  Recall macro    = {rec:.4f}")

    cm = confusion_matrix(test_labels, y_pred, labels=[0, 1, 2])
    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{int_to_label[i]}" for i in [0, 1, 2]],
        columns=[f"pred_{int_to_label[i]}" for i in [0, 1, 2]]
    )
    print("\nConfusion matrix:")
    print(cm_df)

    report = classification_report(
        test_labels,
        y_pred,
        labels=[0, 1, 2],
        target_names=[int_to_label[i] for i in [0, 1, 2]],
        output_dict=True,
        zero_division=0
    )

    print("\nPer-class metrics:")
    for cls_name in label_order:
        print(
            f"  {cls_name}: "
            f"Precision={report[cls_name]['precision']:.4f}, "
            f"Recall={report[cls_name]['recall']:.4f}, "
            f"F1={report[cls_name]['f1-score']:.4f}, "
            f"Support={report[cls_name]['support']:.0f}"
        )

        per_class_results.append({
            "fold": fold,
            "Class": cls_name,
            "Precision": report[cls_name]["precision"],
            "Recall": report[cls_name]["recall"],
            "F1_score": report[cls_name]["f1-score"],
            "Support": report[cls_name]["support"]
        })

    fold_pred_df = pd.DataFrame({
        "fold": fold,
        "test_id": test_ids,
        "true_label": [int_to_label[v] for v in test_labels],
        "pred_label": [int_to_label[v] for v in y_pred],
        "best_train_id": best_train_ids,
        "best_identity": best_identities
    })
    all_fold_predictions.append(fold_pred_df)


# =========================================================
# 8. SAVE RESULTS
# =========================================================
results_df = pd.DataFrame([{
    "Model": "SequenceIdentityNearestNeighbor",
    "Accuracy_mean": np.mean(accs),
    "Accuracy_std": np.std(accs),
    "F1_macro_mean": np.mean(f1s),
    "F1_macro_std": np.std(f1s),
    "Precision_macro_mean": np.mean(precs),
    "Precision_macro_std": np.std(precs),
    "Recall_macro_mean": np.mean(recs),
    "Recall_macro_std": np.std(recs),
}])

per_class_df = pd.DataFrame(per_class_results)
per_class_summary_df = (
    per_class_df.groupby("Class", as_index=False)
    .agg({
        "Precision": ["mean", "std"],
        "Recall": ["mean", "std"],
        "F1_score": ["mean", "std"],
        "Support": "sum"
    })
)

per_class_summary_df.columns = [
    "Class",
    "Precision_mean", "Precision_std",
    "Recall_mean", "Recall_std",
    "F1_score_mean", "F1_score_std",
    "Support_total"
]

print("\n" + "=" * 60)
print("Final results")
print("=" * 60)
print(results_df)

print("\n" + "=" * 60)
print("Per-class summary")
print("=" * 60)
print(per_class_summary_df)

results_df.to_excel(OUTPUT_RESULTS_XLSX, index=False)
results_df.to_csv(OUTPUT_RESULTS_TSV, sep="\t", index=False)

pred_df = pd.concat(all_fold_predictions, ignore_index=True)
pred_df.to_excel(OUTPUT_PRED_XLSX, index=False)

per_class_summary_df.to_excel(OUTPUT_PER_CLASS_XLSX, index=False)

label_map_df = pd.DataFrame({
    "encoded_label": [0, 1, 2],
    "domain_class": label_order
})
label_map_df.to_excel(OUTPUT_LABEL_MAP_XLSX, index=False)

print("\nSaved:")
print(" -", OUTPUT_RESULTS_XLSX)
print(" -", OUTPUT_RESULTS_TSV)
print(" -", OUTPUT_PRED_XLSX)
print(" -", OUTPUT_PER_CLASS_XLSX)
print(" -", OUTPUT_LABEL_MAP_XLSX)

Loaded metadata file: testset_A_Ak_CAL_table.xlsx
Rows: 388
Columns: ['id', 'sequence', 'domain_label']

Final label counts:
domain_label
A_normal    169
CAL         135
Ak           84
Name: count, dtype: int64

Removed exact duplicate sequence+label rows: 0
Rows after deduplication: 388

Encoded classes:
0 -> A_normal
1 -> Ak
2 -> CAL

Fold 1
  Processed 20/78 test sequences
  Processed 40/78 test sequences
  Processed 60/78 test sequences
  Processed 78/78 test sequences

Fold 1 metrics:
  Accuracy        = 0.7821
  F1 macro        = 0.7168
  Precision macro = 0.7196
  Recall macro    = 0.7175

Confusion matrix:
               pred_A_normal  pred_Ak  pred_CAL
true_A_normal             34        0         0
true_Ak                    2        7         8
true_CAL                   0        7        20

Per-class metrics:
  A_normal: Precision=0.9444, Recall=1.0000, F1=0.9714, Support=34
  Ak: Precision=0.5000, Recall=0.4118, F1=0.4516, Support=17
  CAL: Precision=0.7143, Recall=0.740

In [1]:
import warnings
import os
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================

EMBEDDING_FILE = "testset_A_Ak_CAL_embeddings.npy"
METADATA_FILE = "testset_A_Ak_CAL_table.xlsx"

OUTPUT_PREFIX = "testset_A_vs_KAL_binary_classifier"

OUTPUT_OVERALL_XLSX = f"{OUTPUT_PREFIX}_overall_results.xlsx"
OUTPUT_OVERALL_TSV = f"{OUTPUT_PREFIX}_overall_results.tsv"

OUTPUT_PER_CLASS_XLSX = f"{OUTPUT_PREFIX}_per_class_results.xlsx"
OUTPUT_PREDICTIONS_XLSX = f"{OUTPUT_PREFIX}_fold_predictions.xlsx"
OUTPUT_FOLD_METRICS_XLSX = f"{OUTPUT_PREFIX}_individual_fold_metrics.xlsx"
OUTPUT_CONFUSION_XLSX = f"{OUTPUT_PREFIX}_confusion_matrices.xlsx"
OUTPUT_LABEL_MAP_XLSX = f"{OUTPUT_PREFIX}_label_mapping.xlsx"
OUTPUT_FILTERED_METADATA_XLSX = f"{OUTPUT_PREFIX}_filtered_metadata.xlsx"

MODEL_OUTPUT_DIR = "saved_models_A_vs_KAL"

RANDOM_STATE = 42
N_SPLITS = 5

# Expected number of normal A-domain sequences
EXPECTED_A_COUNT = 169


# =========================================================
# 1. LOAD EMBEDDINGS
# =========================================================

X = np.load(EMBEDDING_FILE)

print("=" * 70)
print("Loaded embedding file:", EMBEDDING_FILE)
print("Embedding shape:", X.shape)
print("=" * 70)


# =========================================================
# 2. LOAD METADATA
# =========================================================

file_extension = os.path.splitext(METADATA_FILE)[1].lower()

if file_extension == ".xlsx":
    df = pd.read_excel(METADATA_FILE)

elif file_extension == ".csv":
    df = pd.read_csv(METADATA_FILE)

elif file_extension in {".tsv", ".txt"}:
    df = pd.read_csv(METADATA_FILE, sep="\t")

else:
    raise ValueError(
        f"Unsupported metadata format: {file_extension}\n"
        "Please use .xlsx, .csv, .tsv, or tab-separated .txt."
    )


print("Loaded metadata file:", METADATA_FILE)
print("Original metadata rows:", len(df))
print("Metadata columns:", df.columns.tolist())


if len(df) != X.shape[0]:
    raise ValueError(
        "\nMetadata and embedding dimensions do not match:\n"
        f"  Metadata rows: {len(df)}\n"
        f"  Embedding rows: {X.shape[0]}\n"
        "The metadata rows must be in exactly the same order as the "
        "embedding rows."
    )


# =========================================================
# 3. FIND AND CLEAN ORIGINAL DOMAIN LABEL
# =========================================================

if "domain_label" in df.columns:
    source_label_column = "domain_label"

elif "domain_type" in df.columns:
    source_label_column = "domain_type"

else:
    raise ValueError(
        "Could not find either 'domain_label' or 'domain_type' "
        "in the metadata file.\n"
        f"Available columns: {df.columns.tolist()}"
    )


# Keep original labels for checking
df["original_domain_label"] = df[source_label_column]


def normalize_original_label(value):
    """
    Normalize different possible spellings of A, Ak, and CAL.
    """

    if pd.isna(value):
        return np.nan

    label = str(value).strip().upper()

    label = (
        label
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    # Normal A domain
    if label in {
        "A",
        "A_NORMAL",
        "ANORMAL",
        "NORMAL_A",
        "NORMAL_A_DOMAIN",
        "A_NORMAL_DOMAIN",
        "A_DOMAIN"
    }:
        return "A"

    # Ak domain
    if label in {
        "AK",
        "AK_DOMAIN",
        "AKDOMAIN",
        "AMP",
        "AMP_BINDING",
        "AMPBINDING",
        "AMP_BINDING_DOMAIN"
    }:
        return "Ak"

    # CAL domain
    if label in {
        "CAL",
        "CAL_DOMAIN",
        "CALDOMAIN"
    }:
        return "CAL"

    # Return unmatched labels unchanged so that they can be reported
    return str(value).strip()


df["normalized_original_label"] = (
    df[source_label_column]
    .apply(normalize_original_label)
)


print("\nOriginal normalized label counts:")
print(
    df["normalized_original_label"]
    .value_counts(dropna=False)
    .to_string()
)


# =========================================================
# 4. FILTER TO A, Ak, AND CAL
# =========================================================

target_original_classes = ["A", "Ak", "CAL"]

keep_mask = df["normalized_original_label"].isin(
    target_original_classes
)

removed_df = df.loc[~keep_mask].copy()

if len(removed_df) > 0:
    print("\nWarning: rows with unrecognized labels will be removed:")
    print(
        removed_df["normalized_original_label"]
        .value_counts(dropna=False)
        .to_string()
    )


# Use positional indices because embedding rows correspond to dataframe rows
keep_positions = np.flatnonzero(keep_mask.to_numpy())

df = df.iloc[keep_positions].copy().reset_index(drop=True)
X = X[keep_positions]


if len(df) != X.shape[0]:
    raise ValueError(
        "Filtered metadata rows do not match filtered embedding rows."
    )


# =========================================================
# 5. CREATE BINARY LABEL: A VS KAL
# =========================================================

# A remains A
# Ak and CAL are combined into KAL
binary_label_map = {
    "A": "A",
    "Ak": "KAL",
    "CAL": "KAL"
}

df["domain_label"] = (
    df["normalized_original_label"]
    .map(binary_label_map)
)


if df["domain_label"].isna().any():
    problematic_rows = df[df["domain_label"].isna()]

    raise ValueError(
        "Some filtered labels could not be converted to binary labels:\n"
        f"{problematic_rows['normalized_original_label'].value_counts()}"
    )


print("\n" + "=" * 70)
print("Binary classification:")
print("  Negative class: A")
print("  Positive class: KAL")
print("  KAL contains: Ak + CAL")
print("=" * 70)


print("\nOriginal class counts after filtering:")
print(
    df["normalized_original_label"]
    .value_counts()
    .reindex(["A", "Ak", "CAL"], fill_value=0)
    .to_string()
)


print("\nFinal binary class counts:")
binary_counts = (
    df["domain_label"]
    .value_counts()
    .reindex(["A", "KAL"], fill_value=0)
)

print(binary_counts.to_string())


# Check expected A-domain count
observed_a_count = int(binary_counts["A"])

if observed_a_count == EXPECTED_A_COUNT:
    print(
        f"\nA-domain count check passed: "
        f"{observed_a_count} A-domain sequences."
    )
else:
    print(
        f"\nWARNING: Expected {EXPECTED_A_COUNT} A-domain sequences, "
        f"but found {observed_a_count}."
    )


# Make sure both classes are present
if df["domain_label"].nunique() != 2:
    raise ValueError(
        "Binary classification requires exactly two classes.\n"
        f"Classes found: {sorted(df['domain_label'].unique())}"
    )


# Check whether each class has enough samples for stratified CV
minimum_class_count = int(binary_counts.min())

if minimum_class_count < N_SPLITS:
    raise ValueError(
        f"The smallest class contains only {minimum_class_count} samples, "
        f"which is not enough for {N_SPLITS}-fold stratified "
        "cross-validation."
    )


# =========================================================
# 6. CREATE SEQUENCE IDENTIFIERS
# =========================================================

possible_id_columns = [
    "id",
    "ID",
    "sequence_id",
    "Sequence_ID",
    "name",
    "Name",
    "accession",
    "Accession"
]

id_column = None

for candidate in possible_id_columns:
    if candidate in df.columns:
        id_column = candidate
        break


if id_column is None:
    df["id"] = [
        f"seq_{i + 1:04d}"
        for i in range(len(df))
    ]

    print("\nNo ID column found. Generated sequence IDs.")

elif id_column != "id":
    df["id"] = df[id_column].astype(str)

    print(f"\nUsing '{id_column}' as the sequence ID column.")

else:
    df["id"] = df["id"].astype(str)

    print("\nUsing 'id' as the sequence ID column.")


# =========================================================
# 7. EXPLICIT BINARY ENCODING
# =========================================================

# We explicitly define:
# A   = 0, negative class
# KAL = 1, positive class
label_to_integer = {
    "A": 0,
    "KAL": 1
}

integer_to_label = {
    0: "A",
    1: "KAL"
}

y = (
    df["domain_label"]
    .map(label_to_integer)
    .to_numpy(dtype=int)
)


print("\nEncoded binary labels:")
print("  0 -> A")
print("  1 -> KAL")


# =========================================================
# 8. CHECK EMBEDDINGS
# =========================================================

if X.ndim != 2:
    raise ValueError(
        f"Expected a two-dimensional embedding matrix, "
        f"but the embedding shape is {X.shape}."
    )


if not np.issubdtype(X.dtype, np.number):
    raise ValueError(
        f"Embeddings must be numeric. Current dtype: {X.dtype}"
    )


print("\nFinal dataset:")
print("  Number of sequences:", X.shape[0])
print("  Embedding dimensions:", X.shape[1])
print("  A count:", np.sum(y == 0))
print("  KAL count:", np.sum(y == 1))


# =========================================================
# 9. DEFINE MODELS
# =========================================================

def make_model(model_name, weighted=False):
    """
    Create a new sklearn pipeline for each fold.

    Weighted=True:
      Logistic regression, SVM, and random forest use
      class_weight='balanced'.

      XGBoost receives balanced sample weights during fitting.
    """

    if model_name == "LogisticRegression":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced" if weighted else None,
                    solver="liblinear",
                    random_state=RANDOM_STATE
                )
            )
        ])

    if model_name == "SVM":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight="balanced" if weighted else None,
                    random_state=RANDOM_STATE
                )
            )
        ])

    if model_name == "RandomForest":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=500,
                    class_weight="balanced" if weighted else None,
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ])

    if model_name == "XGBoost":
        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "clf",
                XGBClassifier(
                    n_estimators=500,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ])

    raise ValueError(f"Unknown model: {model_name}")


model_names = [
    "LogisticRegression",
    "SVM",
    "RandomForest",
    "XGBoost"
]


# =========================================================
# 10. HELPER FUNCTION FOR BINARY METRICS
# =========================================================

def calculate_binary_metrics(y_true, y_pred, y_probability):
    """
    Calculate binary metrics.

    KAL is treated as the positive class.
    """

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    accuracy = accuracy_score(y_true, y_pred)

    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        pos_label=1,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    negative_predictive_value = (
        tn / (tn + fn)
        if (tn + fn) > 0
        else np.nan
    )

    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )

    try:
        roc_auc = roc_auc_score(
            y_true,
            y_probability
        )
    except ValueError:
        roc_auc = np.nan

    return {
        "Accuracy": accuracy,
        "Balanced_accuracy": balanced_accuracy,
        "Precision_KAL": precision,
        "Recall_KAL_sensitivity": recall,
        "Specificity_A": specificity,
        "F1_KAL": f1,
        "NPV_A": negative_predictive_value,
        "MCC": mcc,
        "ROC_AUC": roc_auc,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp)
    }


# =========================================================
# 11. SET UP STRATIFIED CROSS-VALIDATION
# =========================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


overall_results = []
fold_metric_results = []
per_class_results = []
confusion_results = []
all_fold_predictions = []


# =========================================================
# 12. CROSS-VALIDATION
# =========================================================

for weighting in [False, True]:

    weight_mode = (
        "weighted"
        if weighting
        else "unweighted"
    )

    print("\n" + "=" * 70)
    print("Running weight mode:", weight_mode)
    print("=" * 70)

    for model_name in model_names:

        print("\nModel:", model_name)

        model_fold_metrics = []

        pooled_y_true = []
        pooled_y_pred = []
        pooled_y_probability = []

        for fold, (train_idx, test_idx) in enumerate(
            cv.split(X, y),
            start=1
        ):

            X_train = X[train_idx]
            X_test = X[test_idx]

            y_train = y[train_idx]
            y_test = y[test_idx]

            model = make_model(
                model_name=model_name,
                weighted=weighting
            )

            # XGBoost uses balanced sample weights
            if model_name == "XGBoost" and weighting:

                training_sample_weights = compute_sample_weight(
                    class_weight="balanced",
                    y=y_train
                )

                model.fit(
                    X_train,
                    y_train,
                    clf__sample_weight=training_sample_weights
                )

            else:
                model.fit(
                    X_train,
                    y_train
                )

            y_pred = model.predict(X_test)

            predicted_probabilities = model.predict_proba(X_test)

            # Probability of positive class KAL, encoded as 1
            model_classes = model.named_steps["clf"].classes_
            positive_class_position = int(
                np.where(model_classes == 1)[0][0]
            )

            y_probability = predicted_probabilities[
                :,
                positive_class_position
            ]

            fold_metrics = calculate_binary_metrics(
                y_true=y_test,
                y_pred=y_pred,
                y_probability=y_probability
            )

            fold_metrics.update({
                "Weight_mode": weight_mode,
                "Model": model_name,
                "Fold": fold,
                "Train_n": len(train_idx),
                "Test_n": len(test_idx),
                "Train_A_n": int(np.sum(y_train == 0)),
                "Train_KAL_n": int(np.sum(y_train == 1)),
                "Test_A_n": int(np.sum(y_test == 0)),
                "Test_KAL_n": int(np.sum(y_test == 1))
            })

            fold_metric_results.append(fold_metrics)
            model_fold_metrics.append(fold_metrics)

            pooled_y_true.extend(y_test.tolist())
            pooled_y_pred.extend(y_pred.tolist())
            pooled_y_probability.extend(
                y_probability.tolist()
            )

            test_metadata = df.iloc[test_idx]

            fold_prediction_df = pd.DataFrame({
                "Weight_mode": weight_mode,
                "Model": model_name,
                "Fold": fold,
                "row_index_in_filtered_dataset": test_idx,
                "id": test_metadata["id"].to_numpy(),
                "original_domain_label":
                    test_metadata[
                        "normalized_original_label"
                    ].to_numpy(),
                "true_binary_label":
                    [
                        integer_to_label[value]
                        for value in y_test
                    ],
                "true_binary_encoded": y_test,
                "predicted_binary_label":
                    [
                        integer_to_label[value]
                        for value in y_pred
                    ],
                "predicted_binary_encoded": y_pred,
                "probability_A": 1.0 - y_probability,
                "probability_KAL": y_probability,
                "correct_prediction": y_test == y_pred
            })

            all_fold_predictions.append(
                fold_prediction_df
            )

            print(
                f"  Fold {fold}: "
                f"ACC={fold_metrics['Accuracy']:.4f}, "
                f"Balanced_ACC="
                f"{fold_metrics['Balanced_accuracy']:.4f}, "
                f"F1_KAL={fold_metrics['F1_KAL']:.4f}, "
                f"Recall_KAL="
                f"{fold_metrics['Recall_KAL_sensitivity']:.4f}, "
                f"Specificity_A="
                f"{fold_metrics['Specificity_A']:.4f}, "
                f"AUC={fold_metrics['ROC_AUC']:.4f}"
            )

        # -------------------------------------------------
        # Summarize metrics across folds
        # -------------------------------------------------

        metric_names = [
            "Accuracy",
            "Balanced_accuracy",
            "Precision_KAL",
            "Recall_KAL_sensitivity",
            "Specificity_A",
            "F1_KAL",
            "NPV_A",
            "MCC",
            "ROC_AUC"
        ]

        overall_row = {
            "Weight_mode": weight_mode,
            "Model": model_name,
            "N_splits": N_SPLITS,
            "Total_sequences": len(y),
            "A_n": int(np.sum(y == 0)),
            "KAL_n": int(np.sum(y == 1))
        }

        for metric_name in metric_names:

            metric_values = np.array([
                row[metric_name]
                for row in model_fold_metrics
            ], dtype=float)

            overall_row[
                f"{metric_name}_mean"
            ] = np.nanmean(metric_values)

            overall_row[
                f"{metric_name}_std"
            ] = np.nanstd(metric_values)

        # Also calculate pooled out-of-fold metrics
        pooled_metrics = calculate_binary_metrics(
            y_true=np.asarray(pooled_y_true),
            y_pred=np.asarray(pooled_y_pred),
            y_probability=np.asarray(
                pooled_y_probability
            )
        )

        for metric_name in metric_names:
            overall_row[
                f"{metric_name}_pooled"
            ] = pooled_metrics[metric_name]

        overall_row["Pooled_TN"] = pooled_metrics["TN"]
        overall_row["Pooled_FP"] = pooled_metrics["FP"]
        overall_row["Pooled_FN"] = pooled_metrics["FN"]
        overall_row["Pooled_TP"] = pooled_metrics["TP"]

        overall_results.append(overall_row)

        # -------------------------------------------------
        # Pooled classification report
        # -------------------------------------------------

        report = classification_report(
            pooled_y_true,
            pooled_y_pred,
            labels=[0, 1],
            target_names=["A", "KAL"],
            output_dict=True,
            zero_division=0
        )

        for class_name in ["A", "KAL"]:
            per_class_results.append({
                "Weight_mode": weight_mode,
                "Model": model_name,
                "Class": class_name,
                "Precision": report[class_name]["precision"],
                "Recall": report[class_name]["recall"],
                "F1_score": report[class_name]["f1-score"],
                "Support": report[class_name]["support"]
            })

        # -------------------------------------------------
        # Pooled confusion matrix
        # -------------------------------------------------

        pooled_cm = confusion_matrix(
            pooled_y_true,
            pooled_y_pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = pooled_cm.ravel()

        confusion_results.append({
            "Weight_mode": weight_mode,
            "Model": model_name,
            "True_A_predicted_A_TN": int(tn),
            "True_A_predicted_KAL_FP": int(fp),
            "True_KAL_predicted_A_FN": int(fn),
            "True_KAL_predicted_KAL_TP": int(tp)
        })


# =========================================================
# 13. CREATE RESULT DATAFRAMES
# =========================================================

overall_df = pd.DataFrame(overall_results)

fold_metrics_df = pd.DataFrame(
    fold_metric_results
)

per_class_df = pd.DataFrame(
    per_class_results
)

confusion_df = pd.DataFrame(
    confusion_results
)

predictions_df = pd.concat(
    all_fold_predictions,
    ignore_index=True
)


# Sort best-performing models first
overall_df = overall_df.sort_values(
    by=[
        "F1_KAL_mean",
        "ROC_AUC_mean",
        "Balanced_accuracy_mean",
        "Accuracy_mean"
    ],
    ascending=False
).reset_index(drop=True)


fold_metrics_df = fold_metrics_df.sort_values(
    by=[
        "Weight_mode",
        "Model",
        "Fold"
    ]
).reset_index(drop=True)


per_class_df = per_class_df.sort_values(
    by=[
        "Weight_mode",
        "Model",
        "Class"
    ]
).reset_index(drop=True)


predictions_df = predictions_df.sort_values(
    by=[
        "Weight_mode",
        "Model",
        "Fold",
        "id"
    ]
).reset_index(drop=True)


# =========================================================
# 14. LABEL-MAPPING TABLE
# =========================================================

label_map_df = pd.DataFrame({
    "original_domain_label": [
        "A",
        "Ak",
        "CAL"
    ],
    "binary_domain_label": [
        "A",
        "KAL",
        "KAL"
    ],
    "binary_encoded_label": [
        0,
        1,
        1
    ],
    "classification_role": [
        "negative class",
        "positive class",
        "positive class"
    ]
})


# =========================================================
# 15. SAVE FILTERED METADATA
# =========================================================

metadata_output_columns = [
    "id",
    "original_domain_label",
    "normalized_original_label",
    "domain_label"
]

additional_columns = [
    column
    for column in df.columns
    if column not in metadata_output_columns
]

filtered_metadata_df = df[
    metadata_output_columns + additional_columns
].copy()


# =========================================================
# 16. SAVE CROSS-VALIDATION RESULTS
# =========================================================

overall_df.to_excel(
    OUTPUT_OVERALL_XLSX,
    index=False
)

overall_df.to_csv(
    OUTPUT_OVERALL_TSV,
    sep="\t",
    index=False
)

per_class_df.to_excel(
    OUTPUT_PER_CLASS_XLSX,
    index=False
)

predictions_df.to_excel(
    OUTPUT_PREDICTIONS_XLSX,
    index=False
)

fold_metrics_df.to_excel(
    OUTPUT_FOLD_METRICS_XLSX,
    index=False
)

confusion_df.to_excel(
    OUTPUT_CONFUSION_XLSX,
    index=False
)

label_map_df.to_excel(
    OUTPUT_LABEL_MAP_XLSX,
    index=False
)

filtered_metadata_df.to_excel(
    OUTPUT_FILTERED_METADATA_XLSX,
    index=False
)


print("\n" + "=" * 70)
print("Saved cross-validation result files")
print("=" * 70)

print(" -", OUTPUT_OVERALL_XLSX)
print(" -", OUTPUT_OVERALL_TSV)
print(" -", OUTPUT_PER_CLASS_XLSX)
print(" -", OUTPUT_PREDICTIONS_XLSX)
print(" -", OUTPUT_FOLD_METRICS_XLSX)
print(" -", OUTPUT_CONFUSION_XLSX)
print(" -", OUTPUT_LABEL_MAP_XLSX)
print(" -", OUTPUT_FILTERED_METADATA_XLSX)


# =========================================================
# 17. DISPLAY OVERALL RESULTS
# =========================================================

display_columns = [
    "Weight_mode",
    "Model",
    "Accuracy_mean",
    "Balanced_accuracy_mean",
    "Precision_KAL_mean",
    "Recall_KAL_sensitivity_mean",
    "Specificity_A_mean",
    "F1_KAL_mean",
    "MCC_mean",
    "ROC_AUC_mean"
]

print("\n" + "=" * 70)
print("Overall cross-validation results")
print("=" * 70)

print(
    overall_df[display_columns]
    .round(4)
    .to_string(index=False)
)


# =========================================================
# 18. TRAIN FINAL WEIGHTED MODELS ON FULL DATASET
# =========================================================

os.makedirs(
    MODEL_OUTPUT_DIR,
    exist_ok=True
)

print("\n" + "=" * 70)
print("Training final weighted models on the full dataset")
print("=" * 70)


saved_model_records = []

for model_name in model_names:

    print("\nTraining final model:", model_name)

    final_model = make_model(
        model_name=model_name,
        weighted=True
    )

    if model_name == "XGBoost":

        full_sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y
        )

        final_model.fit(
            X,
            y,
            clf__sample_weight=full_sample_weights
        )

    else:
        final_model.fit(
            X,
            y
        )

    model_filename = (
        f"{model_name}_weighted_A_vs_KAL_final_model.pkl"
    )

    model_path = os.path.join(
        MODEL_OUTPUT_DIR,
        model_filename
    )

    with open(model_path, "wb") as file:
        pickle.dump(
            final_model,
            file
        )

    saved_model_records.append({
        "Model": model_name,
        "Weight_mode": "weighted",
        "Model_file": model_path,
        "Negative_class": "A",
        "Positive_class": "KAL",
        "A_n": int(np.sum(y == 0)),
        "KAL_n": int(np.sum(y == 1)),
        "Embedding_dimension": X.shape[1]
    })

    print("Saved:", model_path)


# =========================================================
# 19. SAVE LABEL INFORMATION
# =========================================================

label_information = {
    "task": "binary_classification",
    "negative_class": "A",
    "positive_class": "KAL",
    "label_to_integer": {
        "A": 0,
        "KAL": 1
    },
    "integer_to_label": {
        0: "A",
        1: "KAL"
    },
    "original_to_binary": {
        "A": "A",
        "Ak": "KAL",
        "CAL": "KAL"
    }
}


label_information_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "A_vs_KAL_label_information.pkl"
)

with open(label_information_path, "wb") as file:
    pickle.dump(
        label_information,
        file
    )

print("\nSaved:", label_information_path)


# =========================================================
# 20. SAVE DATASET INFORMATION
# =========================================================

dataset_information = {
    "embedding_file": EMBEDDING_FILE,
    "metadata_file": METADATA_FILE,
    "number_of_sequences": int(X.shape[0]),
    "embedding_dimension": int(X.shape[1]),
    "A_count": int(np.sum(y == 0)),
    "KAL_count": int(np.sum(y == 1)),
    "Ak_count": int(
        np.sum(
            df["normalized_original_label"] == "Ak"
        )
    ),
    "CAL_count": int(
        np.sum(
            df["normalized_original_label"] == "CAL"
        )
    ),
    "random_state": RANDOM_STATE,
    "number_of_cv_folds": N_SPLITS
}


dataset_information_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "A_vs_KAL_dataset_information.pkl"
)

with open(dataset_information_path, "wb") as file:
    pickle.dump(
        dataset_information,
        file
    )

print("Saved:", dataset_information_path)


# =========================================================
# 21. SAVE MODEL FILE INDEX
# =========================================================

saved_models_df = pd.DataFrame(
    saved_model_records
)

saved_models_index_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "saved_model_index.xlsx"
)

saved_models_df.to_excel(
    saved_models_index_path,
    index=False
)

print("Saved:", saved_models_index_path)


# =========================================================
# 22. FINISH
# =========================================================

print("\n" + "=" * 70)
print("Binary classification completed")
print("=" * 70)

print("Classification task:")
print("  A = normal A domain")
print("  KAL = Ak + CAL")
print("")
print("Encoding:")
print("  A   = 0")
print("  KAL = 1")
print("")
print("Final sample counts:")
print("  A:", int(np.sum(y == 0)))
print("  Ak:", dataset_information["Ak_count"])
print("  CAL:", dataset_information["CAL_count"])
print("  KAL:", int(np.sum(y == 1)))
print("  Total:", len(y))
print("")
print("Final weighted models are saved in:")
print(" ", MODEL_OUTPUT_DIR)
print("\nDone.")

Loaded embedding file: testset_A_Ak_CAL_embeddings.npy
Embedding shape: (388, 1280)
Loaded metadata file: testset_A_Ak_CAL_table.xlsx
Original metadata rows: 388
Metadata columns: ['id', 'sequence', 'domain_label']

Original normalized label counts:
normalized_original_label
A      169
CAL    135
Ak      84

Binary classification:
  Negative class: A
  Positive class: KAL
  KAL contains: Ak + CAL

Original class counts after filtering:
normalized_original_label
A      169
Ak      84
CAL    135

Final binary class counts:
domain_label
A      169
KAL    219

A-domain count check passed: 169 A-domain sequences.

Using 'id' as the sequence ID column.

Encoded binary labels:
  0 -> A
  1 -> KAL

Final dataset:
  Number of sequences: 388
  Embedding dimensions: 1280
  A count: 169
  KAL count: 219

Running weight mode: unweighted

Model: LogisticRegression
  Fold 1: ACC=1.0000, Balanced_ACC=1.0000, F1_KAL=1.0000, Recall_KAL=1.0000, Specificity_A=1.0000, AUC=1.0000
  Fold 2: ACC=0.9872, Balan

In [2]:
import warnings
import os
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================

EMBEDDING_FILE = "testset_A_Ak_CAL_embeddings.npy"
METADATA_FILE = "testset_A_Ak_CAL_table.xlsx"

OUTPUT_RESULTS = "A_vs_KAL_binary_results.xlsx"
OUTPUT_PREDICTIONS = "A_vs_KAL_binary_predictions.xlsx"

MODEL_OUTPUT_DIR = "saved_models_A_vs_KAL"

RANDOM_STATE = 42
N_SPLITS = 5

# Expected number of normal A-domain sequences
EXPECTED_A_COUNT = 169


# =========================================================
# 1. LOAD EMBEDDINGS
# =========================================================

X = np.load(EMBEDDING_FILE)

print("=" * 60)
print("Loaded embeddings:", EMBEDDING_FILE)
print("Embedding shape:", X.shape)
print("=" * 60)


# =========================================================
# 2. LOAD METADATA
# =========================================================

extension = os.path.splitext(METADATA_FILE)[1].lower()

if extension == ".xlsx":
    df = pd.read_excel(METADATA_FILE)

elif extension == ".csv":
    df = pd.read_csv(METADATA_FILE)

elif extension in {".tsv", ".txt"}:
    df = pd.read_csv(METADATA_FILE, sep="\t")

else:
    raise ValueError(
        "Metadata file must be .xlsx, .csv, .tsv, or .txt"
    )


print("Loaded metadata:", METADATA_FILE)
print("Metadata rows:", len(df))
print("Columns:", df.columns.tolist())


if len(df) != X.shape[0]:
    raise ValueError(
        f"Metadata rows ({len(df)}) do not match "
        f"embedding rows ({X.shape[0]})."
    )


# =========================================================
# 3. FIND LABEL COLUMN
# =========================================================

if "domain_label" in df.columns:
    label_column = "domain_label"

elif "domain_type" in df.columns:
    label_column = "domain_type"

else:
    raise ValueError(
        "Could not find 'domain_label' or 'domain_type' "
        "in the metadata file."
    )


# =========================================================
# 4. NORMALIZE ORIGINAL LABELS
# =========================================================

def normalize_label(value):

    if pd.isna(value):
        return np.nan

    label = str(value).strip().upper()

    label = (
        label
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    # Normal A domain
    if label in {
        "A",
        "A_NORMAL",
        "ANORMAL",
        "NORMAL_A",
        "A_DOMAIN",
        "A_NORMAL_DOMAIN",
        "NORMAL_A_DOMAIN"
    }:
        return "A"

    # Ak domain
    if label in {
        "AK",
        "AK_DOMAIN",
        "AKDOMAIN",
        "AMP",
        "AMP_BINDING",
        "AMPBINDING",
        "AMP_BINDING_DOMAIN"
    }:
        return "Ak"

    # CAL domain
    if label in {
        "CAL",
        "CAL_DOMAIN",
        "CALDOMAIN"
    }:
        return "CAL"

    return str(value).strip()


df["original_label"] = df[label_column]
df["normalized_label"] = df[label_column].apply(normalize_label)


print("\nOriginal normalized label counts:")
print(df["normalized_label"].value_counts(dropna=False))


# =========================================================
# 5. KEEP ONLY A, Ak, AND CAL
# =========================================================

keep_mask = df["normalized_label"].isin(["A", "Ak", "CAL"])

keep_positions = np.flatnonzero(keep_mask.to_numpy())

df = df.iloc[keep_positions].copy().reset_index(drop=True)
X = X[keep_positions]


if len(df) != X.shape[0]:
    raise ValueError(
        "Filtered metadata and embeddings do not match."
    )


# =========================================================
# 6. COMBINE Ak AND CAL AS KAL
# =========================================================

df["binary_label"] = df["normalized_label"].map({
    "A": "A",
    "Ak": "KAL",
    "CAL": "KAL"
})


print("\nOriginal class counts:")
print(df["normalized_label"].value_counts())


print("\nBinary class counts:")
print(df["binary_label"].value_counts())


a_count = int((df["binary_label"] == "A").sum())
kal_count = int((df["binary_label"] == "KAL").sum())


if a_count == EXPECTED_A_COUNT:
    print(f"\nA-domain count confirmed: {a_count}")

else:
    print(
        f"\nWarning: expected {EXPECTED_A_COUNT} A domains, "
        f"but found {a_count}."
    )


print("\nBinary classification definition:")
print("  A   = 0, negative class")
print("  KAL = 1, positive class")
print("  KAL includes Ak and CAL")


# =========================================================
# 7. CREATE IDS
# =========================================================

possible_id_columns = [
    "id",
    "ID",
    "sequence_id",
    "Sequence_ID",
    "name",
    "Name",
    "accession",
    "Accession"
]

id_column = None

for column in possible_id_columns:
    if column in df.columns:
        id_column = column
        break


if id_column is None:
    df["id"] = [
        f"seq_{i + 1:04d}"
        for i in range(len(df))
    ]

elif id_column != "id":
    df["id"] = df[id_column].astype(str)

else:
    df["id"] = df["id"].astype(str)


# =========================================================
# 8. ENCODE BINARY LABELS
# =========================================================

# Explicit encoding:
# A = 0
# KAL = 1

y = df["binary_label"].map({
    "A": 0,
    "KAL": 1
}).to_numpy(dtype=int)


if len(np.unique(y)) != 2:
    raise ValueError(
        "The filtered dataset does not contain both A and KAL."
    )


minimum_class_count = min(
    np.sum(y == 0),
    np.sum(y == 1)
)

if minimum_class_count < N_SPLITS:
    raise ValueError(
        f"The smallest class has only {minimum_class_count} samples. "
        f"This is insufficient for {N_SPLITS}-fold CV."
    )


# =========================================================
# 9. DEFINE MODELS
# =========================================================

def make_model(model_name):
    """
    All models use balanced class weighting.

    KAL is the positive class.
    """

    if model_name == "LogisticRegression":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    solver="liblinear",
                    random_state=RANDOM_STATE
                )
            )
        ])

    if model_name == "SVM":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight="balanced",
                    random_state=RANDOM_STATE
                )
            )
        ])

    if model_name == "RandomForest":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=500,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ])

    if model_name == "XGBoost":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "clf",
                XGBClassifier(
                    n_estimators=500,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ])

    raise ValueError(f"Unknown model: {model_name}")


model_names = [
    "LogisticRegression",
    "SVM",
    "RandomForest",
    "XGBoost"
]


# =========================================================
# 10. STRATIFIED 5-FOLD CROSS-VALIDATION
# =========================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


results = []
all_predictions = []


for model_name in model_names:

    print("\n" + "=" * 60)
    print("Running model:", model_name)
    print("=" * 60)

    # Each sequence receives exactly one out-of-fold prediction
    out_of_fold_predictions = np.zeros(
        len(y),
        dtype=int
    )

    out_of_fold_probabilities = np.zeros(
        len(y),
        dtype=float
    )

    fold_numbers = np.zeros(
        len(y),
        dtype=int
    )

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        model = make_model(model_name)

        # XGBoost does not use sklearn class_weight,
        # so balanced sample weights are supplied manually.
        if model_name == "XGBoost":

            sample_weights = compute_sample_weight(
                class_weight="balanced",
                y=y_train
            )

            model.fit(
                X_train,
                y_train,
                clf__sample_weight=sample_weights
            )

        else:

            model.fit(
                X_train,
                y_train
            )

        y_pred = model.predict(X_test)

        probability_matrix = model.predict_proba(X_test)

        model_classes = model.named_steps["clf"].classes_

        kal_probability_column = int(
            np.where(model_classes == 1)[0][0]
        )

        y_probability = probability_matrix[
            :,
            kal_probability_column
        ]

        out_of_fold_predictions[test_idx] = y_pred
        out_of_fold_probabilities[test_idx] = y_probability
        fold_numbers[test_idx] = fold

        print(
            f"Fold {fold}: "
            f"test sequences = {len(test_idx)}"
        )


    # =====================================================
    # Calculate one set of binary metrics per model
    # =====================================================

    accuracy = accuracy_score(
        y,
        out_of_fold_predictions
    )

    # KAL is positive class
    precision = precision_score(
        y,
        out_of_fold_predictions,
        pos_label=1,
        zero_division=0
    )

    recall = recall_score(
        y,
        out_of_fold_predictions,
        pos_label=1,
        zero_division=0
    )

    f1 = f1_score(
        y,
        out_of_fold_predictions,
        pos_label=1,
        zero_division=0
    )

    auroc = roc_auc_score(
        y,
        out_of_fold_probabilities
    )

    # Confusion matrix:
    #
    #              Predicted A    Predicted KAL
    # True A           TN              FP
    # True KAL         FN              TP

    tn, fp, fn, tp = confusion_matrix(
        y,
        out_of_fold_predictions,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )


    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1": f1,
        "AUROC": auroc
    })


    prediction_df = pd.DataFrame({
        "Model": model_name,
        "Fold": fold_numbers,
        "ID": df["id"],
        "Original_class": df["normalized_label"],
        "True_binary_class": df["binary_label"],
        "Predicted_binary_class": np.where(
            out_of_fold_predictions == 1,
            "KAL",
            "A"
        ),
        "Probability_KAL": out_of_fold_probabilities,
        "Correct": (
            y == out_of_fold_predictions
        )
    })

    all_predictions.append(prediction_df)


    print("\nOut-of-fold results:")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1:          {f1:.4f}")
    print(f"AUROC:       {auroc:.4f}")


# =========================================================
# 11. SAVE SIMPLE RESULT TABLE
# =========================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=[
        "F1",
        "AUROC",
        "Accuracy"
    ],
    ascending=False
).reset_index(drop=True)


# Round displayed Excel values
results_df = results_df.round(4)

results_df.to_excel(
    OUTPUT_RESULTS,
    index=False
)


predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

predictions_df.to_excel(
    OUTPUT_PREDICTIONS,
    index=False
)


print("\n" + "=" * 60)
print("Final binary-classification results")
print("=" * 60)

print(results_df.to_string(index=False))


print("\nSaved:")
print(" -", OUTPUT_RESULTS)
print(" -", OUTPUT_PREDICTIONS)


# =========================================================
# 12. TRAIN FINAL MODELS ON ALL DATA
# =========================================================

os.makedirs(
    MODEL_OUTPUT_DIR,
    exist_ok=True
)


print("\n" + "=" * 60)
print("Training final balanced models on all data")
print("=" * 60)


for model_name in model_names:

    final_model = make_model(model_name)

    if model_name == "XGBoost":

        full_sample_weights = compute_sample_weight(
            class_weight="balanced",
            y=y
        )

        final_model.fit(
            X,
            y,
            clf__sample_weight=full_sample_weights
        )

    else:

        final_model.fit(
            X,
            y
        )

    model_path = os.path.join(
        MODEL_OUTPUT_DIR,
        f"{model_name}_A_vs_KAL.pkl"
    )

    with open(model_path, "wb") as file:
        pickle.dump(final_model, file)

    print("Saved:", model_path)


# =========================================================
# 13. SAVE LABEL INFORMATION
# =========================================================

label_information = {
    "A": 0,
    "KAL": 1,
    "positive_class": "KAL",
    "negative_class": "A",
    "KAL_definition": "Ak + CAL"
}


label_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "A_vs_KAL_label_information.pkl"
)

with open(label_path, "wb") as file:
    pickle.dump(label_information, file)


print("Saved:", label_path)


# =========================================================
# 14. FINISH
# =========================================================

print("\nDone.")

print("\nMain file to examine:")
print(" ", OUTPUT_RESULTS)

print("\nMetric definitions:")
print("  Accuracy    = overall correct classification")
print("  Precision   = precision for KAL")
print("  Recall      = sensitivity for KAL")
print("  Specificity = correctly identified A sequences")
print("  F1          = F1 score for KAL")
print("  AUROC       = A-versus-KAL discrimination")

Loaded embeddings: testset_A_Ak_CAL_embeddings.npy
Embedding shape: (388, 1280)
Loaded metadata: testset_A_Ak_CAL_table.xlsx
Metadata rows: 388
Columns: ['id', 'sequence', 'domain_label']

Original normalized label counts:
normalized_label
A      169
CAL    135
Ak      84
Name: count, dtype: int64

Original class counts:
normalized_label
A      169
CAL    135
Ak      84
Name: count, dtype: int64

Binary class counts:
binary_label
KAL    219
A      169
Name: count, dtype: int64

A-domain count confirmed: 169

Binary classification definition:
  A   = 0, negative class
  KAL = 1, positive class
  KAL includes Ak and CAL

Running model: LogisticRegression
Fold 1: test sequences = 78
Fold 2: test sequences = 78
Fold 3: test sequences = 78
Fold 4: test sequences = 77
Fold 5: test sequences = 77

Out-of-fold results:
Accuracy:    0.9897
Precision:   0.9909
Recall:      0.9909
Specificity: 0.9882
F1:          0.9909
AUROC:       0.9991

Running model: SVM
Fold 1: test sequences = 78
Fold 2: t

In [3]:
import warnings
import os
import pickle

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier


warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================

EMBEDDING_FILE = "testset_A_Ak_CAL_embeddings.npy"
METADATA_FILE = "testset_A_Ak_CAL_table.xlsx"

OUTPUT_RESULTS = "A_vs_KAL_binary_results_unweighted.xlsx"
OUTPUT_PREDICTIONS = "A_vs_KAL_binary_predictions_unweighted.xlsx"

MODEL_OUTPUT_DIR = "saved_models_A_vs_KAL_unweighted"

RANDOM_STATE = 42
N_SPLITS = 5

EXPECTED_A_COUNT = 169
EXPECTED_KAL_COUNT = 219


# =========================================================
# 1. LOAD EMBEDDINGS
# =========================================================

X = np.load(EMBEDDING_FILE)

print("=" * 70)
print("Loaded embedding file:", EMBEDDING_FILE)
print("Embedding shape:", X.shape)
print("=" * 70)


# =========================================================
# 2. LOAD METADATA
# =========================================================

extension = os.path.splitext(METADATA_FILE)[1].lower()

if extension == ".xlsx":
    df = pd.read_excel(METADATA_FILE)

elif extension == ".csv":
    df = pd.read_csv(METADATA_FILE)

elif extension in {".tsv", ".txt"}:
    df = pd.read_csv(METADATA_FILE, sep="\t")

else:
    raise ValueError(
        "Metadata file must be .xlsx, .csv, .tsv, or .txt"
    )


print("Loaded metadata file:", METADATA_FILE)
print("Metadata rows:", len(df))
print("Metadata columns:", df.columns.tolist())


if len(df) != X.shape[0]:
    raise ValueError(
        f"Metadata rows ({len(df)}) do not match "
        f"embedding rows ({X.shape[0]})."
    )


# =========================================================
# 3. FIND LABEL COLUMN
# =========================================================

if "domain_label" in df.columns:
    label_column = "domain_label"

elif "domain_type" in df.columns:
    label_column = "domain_type"

else:
    raise ValueError(
        "Could not find 'domain_label' or 'domain_type' "
        "in the metadata file."
    )


print("Using label column:", label_column)


# =========================================================
# 4. NORMALIZE LABELS
# =========================================================

def normalize_label(value):
    """
    Convert different label spellings into:
      A
      Ak
      CAL
    """

    if pd.isna(value):
        return np.nan

    label = str(value).strip().upper()

    label = (
        label
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    # Normal A domain
    if label in {
        "A",
        "A_NORMAL",
        "ANORMAL",
        "NORMAL_A",
        "A_DOMAIN",
        "A_NORMAL_DOMAIN",
        "NORMAL_A_DOMAIN"
    }:
        return "A"

    # Ak domain
    if label in {
        "AK",
        "AK_DOMAIN",
        "AKDOMAIN",
        "AMP",
        "AMP_BINDING",
        "AMPBINDING",
        "AMP_BINDING_DOMAIN"
    }:
        return "Ak"

    # CAL domain
    if label in {
        "CAL",
        "CAL_DOMAIN",
        "CALDOMAIN"
    }:
        return "CAL"

    return str(value).strip()


df["original_label"] = df[label_column]

df["normalized_label"] = (
    df[label_column]
    .apply(normalize_label)
)


print("\nNormalized original label counts:")
print(
    df["normalized_label"]
    .value_counts(dropna=False)
)


# =========================================================
# 5. KEEP ONLY A, Ak, AND CAL
# =========================================================

keep_mask = df["normalized_label"].isin(
    ["A", "Ak", "CAL"]
)

removed_count = int((~keep_mask).sum())

if removed_count > 0:
    print(
        f"\nWarning: removing {removed_count} rows "
        "with labels other than A, Ak, or CAL."
    )

    print(
        df.loc[~keep_mask, "normalized_label"]
        .value_counts(dropna=False)
    )


keep_positions = np.flatnonzero(
    keep_mask.to_numpy()
)

df = (
    df.iloc[keep_positions]
    .copy()
    .reset_index(drop=True)
)

X = X[keep_positions]


if len(df) != X.shape[0]:
    raise ValueError(
        "Filtered metadata rows do not match "
        "filtered embedding rows."
    )


# =========================================================
# 6. COMBINE Ak AND CAL INTO KAL
# =========================================================

df["binary_label"] = (
    df["normalized_label"]
    .map({
        "A": "A",
        "Ak": "KAL",
        "CAL": "KAL"
    })
)


print("\nOriginal class counts after filtering:")
print(
    df["normalized_label"]
    .value_counts()
    .reindex(
        ["A", "Ak", "CAL"],
        fill_value=0
    )
)


print("\nBinary class counts:")
print(
    df["binary_label"]
    .value_counts()
    .reindex(
        ["A", "KAL"],
        fill_value=0
    )
)


a_count = int(
    (df["binary_label"] == "A").sum()
)

kal_count = int(
    (df["binary_label"] == "KAL").sum()
)


if a_count == EXPECTED_A_COUNT:
    print(
        f"\nA count confirmed: {a_count}"
    )

else:
    print(
        f"\nWarning: expected {EXPECTED_A_COUNT} A sequences, "
        f"but found {a_count}."
    )


if kal_count == EXPECTED_KAL_COUNT:
    print(
        f"KAL count confirmed: {kal_count}"
    )

else:
    print(
        f"Warning: expected {EXPECTED_KAL_COUNT} KAL sequences, "
        f"but found {kal_count}."
    )


print("\nBinary classification:")
print("  A   = negative class, encoded as 0")
print("  KAL = positive class, encoded as 1")
print("  KAL = Ak + CAL")
print("  No class weighting is used")


# =========================================================
# 7. CREATE SEQUENCE IDS
# =========================================================

possible_id_columns = [
    "id",
    "ID",
    "sequence_id",
    "Sequence_ID",
    "name",
    "Name",
    "accession",
    "Accession"
]

id_column = None

for candidate in possible_id_columns:
    if candidate in df.columns:
        id_column = candidate
        break


if id_column is None:

    df["id"] = [
        f"seq_{i + 1:04d}"
        for i in range(len(df))
    ]

    print("\nNo ID column found.")
    print("Generated IDs such as seq_0001.")

elif id_column == "id":

    df["id"] = df["id"].astype(str)

    print("\nUsing 'id' as the sequence ID column.")

else:

    df["id"] = df[id_column].astype(str)

    print(
        f"\nUsing '{id_column}' as the sequence ID column."
    )


# =========================================================
# 8. ENCODE BINARY LABELS
# =========================================================

label_to_integer = {
    "A": 0,
    "KAL": 1
}

integer_to_label = {
    0: "A",
    1: "KAL"
}


y = (
    df["binary_label"]
    .map(label_to_integer)
    .to_numpy(dtype=int)
)


if len(np.unique(y)) != 2:
    raise ValueError(
        "The dataset must contain both A and KAL."
    )


minimum_class_count = min(
    np.sum(y == 0),
    np.sum(y == 1)
)


if minimum_class_count < N_SPLITS:
    raise ValueError(
        f"The smallest class contains only "
        f"{minimum_class_count} samples, "
        f"which is insufficient for {N_SPLITS}-fold CV."
    )


print("\nFinal dataset:")
print("  Total sequences:", len(y))
print("  A sequences:", np.sum(y == 0))
print("  KAL sequences:", np.sum(y == 1))
print("  Embedding dimension:", X.shape[1])


# =========================================================
# 9. DEFINE UNWEIGHTED MODELS
# =========================================================

def make_model(model_name):
    """
    Create an unweighted model.

    No class_weight and no sample weights are used.
    """

    if model_name == "LogisticRegression":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight=None,
                    solver="liblinear",
                    random_state=RANDOM_STATE
                )
            )
        ])

    if model_name == "SVM":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "scaler",
                StandardScaler()
            ),
            (
                "clf",
                SVC(
                    kernel="rbf",
                    probability=True,
                    class_weight=None,
                    random_state=RANDOM_STATE
                )
            )
        ])

    if model_name == "RandomForest":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "clf",
                RandomForestClassifier(
                    n_estimators=500,
                    class_weight=None,
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ])

    if model_name == "XGBoost":

        return Pipeline([
            (
                "imputer",
                SimpleImputer(strategy="mean")
            ),
            (
                "clf",
                XGBClassifier(
                    n_estimators=500,
                    max_depth=6,
                    learning_rate=0.05,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    random_state=RANDOM_STATE,
                    n_jobs=-1
                )
            )
        ])

    raise ValueError(
        f"Unknown model name: {model_name}"
    )


model_names = [
    "LogisticRegression",
    "SVM",
    "RandomForest",
    "XGBoost"
]


# =========================================================
# 10. SET UP STRATIFIED 5-FOLD CV
# =========================================================

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)


results = []
all_prediction_tables = []


# =========================================================
# 11. RUN CROSS-VALIDATION
# =========================================================

for model_name in model_names:

    print("\n" + "=" * 70)
    print("Running model:", model_name)
    print("=" * 70)

    # Every sequence gets exactly one prediction from a model
    # that was not trained on that sequence.
    out_of_fold_predictions = np.zeros(
        len(y),
        dtype=int
    )

    out_of_fold_probabilities = np.zeros(
        len(y),
        dtype=float
    )

    fold_assignments = np.zeros(
        len(y),
        dtype=int
    )


    for fold, (train_idx, test_idx) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        model = make_model(model_name)

        # No class weighting and no sample weighting
        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(X_test)

        probability_matrix = model.predict_proba(
            X_test
        )

        model_classes = (
            model.named_steps["clf"]
            .classes_
        )

        kal_probability_column = int(
            np.where(model_classes == 1)[0][0]
        )

        probability_kal = probability_matrix[
            :,
            kal_probability_column
        ]

        out_of_fold_predictions[test_idx] = y_pred

        out_of_fold_probabilities[
            test_idx
        ] = probability_kal

        fold_assignments[test_idx] = fold


        fold_accuracy = accuracy_score(
            y_test,
            y_pred
        )

        print(
            f"Fold {fold}: "
            f"train={len(train_idx)}, "
            f"test={len(test_idx)}, "
            f"accuracy={fold_accuracy:.4f}"
        )


    # =====================================================
    # 12. CALCULATE ONE RESULT ROW PER MODEL
    # =====================================================

    accuracy = accuracy_score(
        y,
        out_of_fold_predictions
    )


    # KAL is the positive class
    precision = precision_score(
        y,
        out_of_fold_predictions,
        pos_label=1,
        zero_division=0
    )


    recall = recall_score(
        y,
        out_of_fold_predictions,
        pos_label=1,
        zero_division=0
    )


    f1 = f1_score(
        y,
        out_of_fold_predictions,
        pos_label=1,
        zero_division=0
    )


    auroc = roc_auc_score(
        y,
        out_of_fold_probabilities
    )


    # Confusion matrix:
    #
    #                   Predicted A    Predicted KAL
    # True A                TN              FP
    # True KAL              FN              TP

    tn, fp, fn, tp = confusion_matrix(
        y,
        out_of_fold_predictions,
        labels=[0, 1]
    ).ravel()


    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )


    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1": f1,
        "AUROC": auroc
    })


    prediction_table = pd.DataFrame({
        "Model": model_name,
        "Fold": fold_assignments,
        "ID": df["id"],
        "Original_class": df["normalized_label"],
        "True_binary_class": df["binary_label"],
        "Predicted_binary_class": np.where(
            out_of_fold_predictions == 1,
            "KAL",
            "A"
        ),
        "Probability_KAL": out_of_fold_probabilities,
        "Correct": (
            y == out_of_fold_predictions
        )
    })


    all_prediction_tables.append(
        prediction_table
    )


    print("\nBinary classification metrics:")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Precision:   {precision:.4f}")
    print(f"Recall:      {recall:.4f}")
    print(f"Specificity: {specificity:.4f}")
    print(f"F1:          {f1:.4f}")
    print(f"AUROC:       {auroc:.4f}")

    print("\nConfusion matrix counts:")
    print("  True A predicted A:", tn)
    print("  True A predicted KAL:", fp)
    print("  True KAL predicted A:", fn)
    print("  True KAL predicted KAL:", tp)


# =========================================================
# 13. SAVE SIMPLE RESULT TABLE
# =========================================================

results_df = pd.DataFrame(results)


results_df = (
    results_df
    .sort_values(
        by=[
            "F1",
            "AUROC",
            "Accuracy"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)


results_df = results_df.round(4)


results_df.to_excel(
    OUTPUT_RESULTS,
    index=False
)


predictions_df = pd.concat(
    all_prediction_tables,
    ignore_index=True
)


predictions_df.to_excel(
    OUTPUT_PREDICTIONS,
    index=False
)


print("\n" + "=" * 70)
print("Final A-versus-KAL results")
print("=" * 70)

print(
    results_df.to_string(index=False)
)


print("\nSaved result files:")
print(" -", OUTPUT_RESULTS)
print(" -", OUTPUT_PREDICTIONS)


# =========================================================
# 14. TRAIN FINAL UNWEIGHTED MODELS ON ALL DATA
# =========================================================

os.makedirs(
    MODEL_OUTPUT_DIR,
    exist_ok=True
)


print("\n" + "=" * 70)
print("Training final unweighted models on all data")
print("=" * 70)


for model_name in model_names:

    print("\nTraining:", model_name)

    final_model = make_model(
        model_name
    )

    # No sample weights
    final_model.fit(
        X,
        y
    )

    model_path = os.path.join(
        MODEL_OUTPUT_DIR,
        f"{model_name}_A_vs_KAL_unweighted.pkl"
    )

    with open(model_path, "wb") as file:
        pickle.dump(
            final_model,
            file
        )

    print("Saved:", model_path)


# =========================================================
# 15. SAVE LABEL INFORMATION
# =========================================================

label_information = {
    "task": "binary classification",
    "negative_class": "A",
    "positive_class": "KAL",
    "A_encoded_label": 0,
    "KAL_encoded_label": 1,
    "KAL_definition": "Ak + CAL",
    "class_weighting": "none",
    "A_count": a_count,
    "KAL_count": kal_count
}


label_path = os.path.join(
    MODEL_OUTPUT_DIR,
    "A_vs_KAL_label_information.pkl"
)


with open(label_path, "wb") as file:
    pickle.dump(
        label_information,
        file
    )


print("\nSaved:", label_path)


# =========================================================
# 16. FINISH
# =========================================================

print("\n" + "=" * 70)
print("Done")
print("=" * 70)

print("\nMain result file:")
print(" ", OUTPUT_RESULTS)

print("\nThe result table contains:")
print("  Model")
print("  Accuracy")
print("  Precision")
print("  Recall")
print("  Specificity")
print("  F1")
print("  AUROC")

print("\nMetric interpretation:")
print("  Positive class = KAL")
print("  Negative class = A")
print("  Precision = precision for KAL")
print("  Recall = sensitivity for KAL")
print("  Specificity = fraction of A correctly identified")
print("  F1 = F1 score for KAL")
print("  AUROC = overall separation of A and KAL")

Loaded embedding file: testset_A_Ak_CAL_embeddings.npy
Embedding shape: (388, 1280)
Loaded metadata file: testset_A_Ak_CAL_table.xlsx
Metadata rows: 388
Metadata columns: ['id', 'sequence', 'domain_label']
Using label column: domain_label

Normalized original label counts:
normalized_label
A      169
CAL    135
Ak      84
Name: count, dtype: int64

Original class counts after filtering:
normalized_label
A      169
Ak      84
CAL    135
Name: count, dtype: int64

Binary class counts:
binary_label
A      169
KAL    219
Name: count, dtype: int64

A count confirmed: 169
KAL count confirmed: 219

Binary classification:
  A   = negative class, encoded as 0
  KAL = positive class, encoded as 1
  KAL = Ak + CAL
  No class weighting is used

Using 'id' as the sequence ID column.

Final dataset:
  Total sequences: 388
  A sequences: 169
  KAL sequences: 219
  Embedding dimension: 1280

Running model: LogisticRegression
Fold 1: train=310, test=78, accuracy=1.0000
Fold 2: train=310, test=78, accur

In [4]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")


# =========================================================
# 0. SETTINGS
# =========================================================

EMBEDDING_FILE = "testset_A_Ak_CAL_embeddings.npy"
METADATA_FILE = "testset_A_Ak_CAL_table.xlsx"

OUTPUT_DIR = "A_vs_KAL_embedding_plots"

RANDOM_STATE = 42

# Plot settings
POINT_SIZE = 45
POINT_ALPHA = 0.75
FIGURE_WIDTH = 8
FIGURE_HEIGHT = 7
DPI = 600

# Set to True to label every point with its sequence ID.
# For 388 sequences, False is recommended.
SHOW_SEQUENCE_LABELS = False


# =========================================================
# 1. CREATE OUTPUT DIRECTORY
# =========================================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================================================
# 2. LOAD ESM EMBEDDINGS
# =========================================================

X = np.load(EMBEDDING_FILE)

print("=" * 70)
print("Loaded embedding file:", EMBEDDING_FILE)
print("Embedding shape:", X.shape)
print("=" * 70)


if X.ndim != 2:
    raise ValueError(
        f"Expected a 2D embedding matrix, but found shape {X.shape}."
    )


# =========================================================
# 3. LOAD METADATA
# =========================================================

extension = os.path.splitext(METADATA_FILE)[1].lower()

if extension == ".xlsx":
    df = pd.read_excel(METADATA_FILE)

elif extension == ".csv":
    df = pd.read_csv(METADATA_FILE)

elif extension in {".tsv", ".txt"}:
    df = pd.read_csv(METADATA_FILE, sep="\t")

else:
    raise ValueError(
        "Metadata file must be .xlsx, .csv, .tsv, or .txt."
    )


print("Loaded metadata file:", METADATA_FILE)
print("Metadata rows:", len(df))
print("Metadata columns:", df.columns.tolist())


if len(df) != X.shape[0]:
    raise ValueError(
        f"Metadata rows ({len(df)}) do not match "
        f"embedding rows ({X.shape[0]})."
    )


# =========================================================
# 4. FIND LABEL COLUMN
# =========================================================

if "domain_label" in df.columns:
    label_column = "domain_label"

elif "domain_type" in df.columns:
    label_column = "domain_type"

else:
    raise ValueError(
        "Could not find 'domain_label' or 'domain_type' "
        "in the metadata file."
    )


print("Using label column:", label_column)


# =========================================================
# 5. NORMALIZE LABELS
# =========================================================

def normalize_label(value):
    """
    Normalize labels into:
      A
      Ak
      CAL
    """

    if pd.isna(value):
        return np.nan

    label = str(value).strip().upper()

    label = (
        label
        .replace("-", "_")
        .replace(" ", "_")
        .replace("/", "_")
    )

    if label in {
        "A",
        "A_NORMAL",
        "ANORMAL",
        "NORMAL_A",
        "A_DOMAIN",
        "A_NORMAL_DOMAIN",
        "NORMAL_A_DOMAIN"
    }:
        return "A"

    if label in {
        "AK",
        "AK_DOMAIN",
        "AKDOMAIN",
        "AMP",
        "AMP_BINDING",
        "AMPBINDING",
        "AMP_BINDING_DOMAIN"
    }:
        return "Ak"

    if label in {
        "CAL",
        "CAL_DOMAIN",
        "CALDOMAIN"
    }:
        return "CAL"

    return str(value).strip()


df["original_label"] = df[label_column]

df["normalized_label"] = (
    df[label_column]
    .apply(normalize_label)
)


print("\nNormalized label counts:")
print(df["normalized_label"].value_counts(dropna=False))


# =========================================================
# 6. KEEP ONLY A, Ak, AND CAL
# =========================================================

keep_mask = df["normalized_label"].isin(
    ["A", "Ak", "CAL"]
)

removed_count = int((~keep_mask).sum())

if removed_count > 0:
    print(
        f"\nRemoving {removed_count} rows with labels "
        "other than A, Ak, or CAL."
    )


keep_positions = np.flatnonzero(
    keep_mask.to_numpy()
)

df = (
    df.iloc[keep_positions]
    .copy()
    .reset_index(drop=True)
)

X = X[keep_positions]


if len(df) != X.shape[0]:
    raise ValueError(
        "Filtered metadata and embedding rows do not match."
    )


# =========================================================
# 7. COMBINE Ak AND CAL INTO KAL
# =========================================================

df["binary_label"] = df["normalized_label"].map({
    "A": "A",
    "Ak": "KAL",
    "CAL": "KAL"
})


print("\nBinary class counts:")
print(df["binary_label"].value_counts())


a_count = int(
    (df["binary_label"] == "A").sum()
)

kal_count = int(
    (df["binary_label"] == "KAL").sum()
)


print("\nA count:", a_count)
print("KAL count:", kal_count)
print("Total:", len(df))


# =========================================================
# 8. CREATE SEQUENCE IDS
# =========================================================

possible_id_columns = [
    "id",
    "ID",
    "sequence_id",
    "Sequence_ID",
    "name",
    "Name",
    "accession",
    "Accession"
]

id_column = None

for candidate in possible_id_columns:
    if candidate in df.columns:
        id_column = candidate
        break


if id_column is None:
    df["plot_id"] = [
        f"seq_{i + 1:04d}"
        for i in range(len(df))
    ]

else:
    df["plot_id"] = df[id_column].astype(str)


# =========================================================
# 9. HANDLE MISSING OR INFINITE EMBEDDING VALUES
# =========================================================

X = np.asarray(X, dtype=float)

X[~np.isfinite(X)] = np.nan

column_means = np.nanmean(X, axis=0)

nan_rows, nan_columns = np.where(np.isnan(X))

X[nan_rows, nan_columns] = column_means[nan_columns]


if np.isnan(X).any():
    raise ValueError(
        "Embedding matrix still contains missing values."
    )


# =========================================================
# 10. STANDARDIZE EMBEDDINGS
# =========================================================

# Scaling is useful for PCA and generally safe for visualization.
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)


# =========================================================
# 11. COLOR AND MARKER SETTINGS
# =========================================================

class_order = ["A", "KAL"]

class_colors = {
    "A": "#377eb8",
    "KAL": "#e41a1c"
}

class_markers = {
    "A": "o",
    "KAL": "^"
}


# =========================================================
# 12. GENERAL SCATTER-PLOT FUNCTION
# =========================================================

def plot_embedding(
    coordinates,
    x_column,
    y_column,
    x_label,
    y_label,
    title,
    output_filename
):
    """
    Create one A-versus-KAL embedding scatter plot.
    """

    plot_df = df.copy()

    plot_df[x_column] = coordinates[:, 0]
    plot_df[y_column] = coordinates[:, 1]

    plt.figure(
        figsize=(FIGURE_WIDTH, FIGURE_HEIGHT)
    )

    for class_name in class_order:

        subset = plot_df[
            plot_df["binary_label"] == class_name
        ]

        plt.scatter(
            subset[x_column],
            subset[y_column],
            s=POINT_SIZE,
            alpha=POINT_ALPHA,
            marker=class_markers[class_name],
            color=class_colors[class_name],
            edgecolors="black",
            linewidths=0.35,
            label=f"{class_name} (n={len(subset)})"
        )

        if SHOW_SEQUENCE_LABELS:

            for _, row in subset.iterrows():

                plt.annotate(
                    row["plot_id"],
                    (
                        row[x_column],
                        row[y_column]
                    ),
                    fontsize=5,
                    alpha=0.7,
                    xytext=(2, 2),
                    textcoords="offset points"
                )

    plt.xlabel(
        x_label,
        fontsize=13
    )

    plt.ylabel(
        y_label,
        fontsize=13
    )

    plt.title(
        title,
        fontsize=15,
        fontweight="bold"
    )

    plt.legend(
        frameon=True,
        fontsize=11,
        loc="best"
    )

    plt.xticks(fontsize=11)
    plt.yticks(fontsize=11)

    plt.grid(
        alpha=0.2,
        linestyle="--"
    )

    plt.tight_layout()

    output_path = os.path.join(
        OUTPUT_DIR,
        output_filename
    )

    plt.savefig(
        output_path,
        dpi=DPI,
        bbox_inches="tight"
    )

    plt.close()

    print("Saved:", output_path)

    return plot_df


# =========================================================
# 13. PCA PLOT
# =========================================================

print("\n" + "=" * 70)
print("Running PCA")
print("=" * 70)


pca = PCA(
    n_components=2,
    random_state=RANDOM_STATE
)

pca_coordinates = pca.fit_transform(
    X_scaled
)


pc1_variance = (
    pca.explained_variance_ratio_[0] * 100
)

pc2_variance = (
    pca.explained_variance_ratio_[1] * 100
)


pca_df = plot_embedding(
    coordinates=pca_coordinates,
    x_column="PC1",
    y_column="PC2",
    x_label=f"PC1 ({pc1_variance:.2f}% variance)",
    y_label=f"PC2 ({pc2_variance:.2f}% variance)",
    title="PCA of ESM Embeddings: A versus KAL",
    output_filename="A_vs_KAL_ESM_PCA.png"
)


pca_output_table = os.path.join(
    OUTPUT_DIR,
    "A_vs_KAL_ESM_PCA_coordinates.xlsx"
)

pca_df[
    [
        "plot_id",
        "original_label",
        "normalized_label",
        "binary_label",
        "PC1",
        "PC2"
    ]
].to_excel(
    pca_output_table,
    index=False
)

print("Saved:", pca_output_table)


# =========================================================
# 14. UMAP PLOT
# =========================================================

print("\n" + "=" * 70)
print("Running UMAP")
print("=" * 70)


try:
    import umap

except ImportError:
    raise ImportError(
        "\nUMAP is not installed.\n"
        "Install it using:\n"
        "  pip install umap-learn\n"
        "or:\n"
        "  conda install -c conda-forge umap-learn"
    )


umap_model = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=RANDOM_STATE
)

umap_coordinates = umap_model.fit_transform(
    X_scaled
)


umap_df = plot_embedding(
    coordinates=umap_coordinates,
    x_column="UMAP1",
    y_column="UMAP2",
    x_label="UMAP 1",
    y_label="UMAP 2",
    title="UMAP of ESM Embeddings: A versus KAL",
    output_filename="A_vs_KAL_ESM_UMAP.png"
)


umap_output_table = os.path.join(
    OUTPUT_DIR,
    "A_vs_KAL_ESM_UMAP_coordinates.xlsx"
)

umap_df[
    [
        "plot_id",
        "original_label",
        "normalized_label",
        "binary_label",
        "UMAP1",
        "UMAP2"
    ]
].to_excel(
    umap_output_table,
    index=False
)

print("Saved:", umap_output_table)


# =========================================================
# 15. t-SNE PLOT
# =========================================================

print("\n" + "=" * 70)
print("Running t-SNE")
print("=" * 70)


# PCA reduction before t-SNE improves speed and reduces noise.
initial_pca_dimensions = min(
    50,
    X_scaled.shape[1],
    X_scaled.shape[0] - 1
)


pca_for_tsne = PCA(
    n_components=initial_pca_dimensions,
    random_state=RANDOM_STATE
)

X_tsne_input = pca_for_tsne.fit_transform(
    X_scaled
)


# Perplexity must be less than the sample count.
tsne_perplexity = min(
    30,
    max(5, (len(df) - 1) // 3)
)


print(
    "t-SNE input dimensions:",
    initial_pca_dimensions
)

print(
    "t-SNE perplexity:",
    tsne_perplexity
)


tsne_model = TSNE(
    n_components=2,
    perplexity=tsne_perplexity,
    learning_rate="auto",
    init="pca",
    max_iter=2000,
    metric="euclidean",
    random_state=RANDOM_STATE
)

tsne_coordinates = tsne_model.fit_transform(
    X_tsne_input
)


tsne_df = plot_embedding(
    coordinates=tsne_coordinates,
    x_column="tSNE1",
    y_column="tSNE2",
    x_label="t-SNE 1",
    y_label="t-SNE 2",
    title="t-SNE of ESM Embeddings: A versus KAL",
    output_filename="A_vs_KAL_ESM_tSNE.png"
)


tsne_output_table = os.path.join(
    OUTPUT_DIR,
    "A_vs_KAL_ESM_tSNE_coordinates.xlsx"
)

tsne_df[
    [
        "plot_id",
        "original_label",
        "normalized_label",
        "binary_label",
        "tSNE1",
        "tSNE2"
    ]
].to_excel(
    tsne_output_table,
    index=False
)

print("Saved:", tsne_output_table)


# =========================================================
# 16. SAVE ALL COORDINATES IN ONE EXCEL FILE
# =========================================================

combined_coordinates = pd.DataFrame({
    "ID": df["plot_id"],
    "Original_class": df["normalized_label"],
    "Binary_class": df["binary_label"],
    "PC1": pca_coordinates[:, 0],
    "PC2": pca_coordinates[:, 1],
    "UMAP1": umap_coordinates[:, 0],
    "UMAP2": umap_coordinates[:, 1],
    "tSNE1": tsne_coordinates[:, 0],
    "tSNE2": tsne_coordinates[:, 1]
})


combined_output = os.path.join(
    OUTPUT_DIR,
    "A_vs_KAL_ESM_all_embedding_coordinates.xlsx"
)

combined_coordinates.to_excel(
    combined_output,
    index=False
)

print("Saved:", combined_output)


# =========================================================
# 17. FINISH
# =========================================================

print("\n" + "=" * 70)
print("Finished")
print("=" * 70)

print("\nBinary classes:")
print(f"  A: {a_count}")
print(f"  KAL: {kal_count}")
print("  KAL = Ak + CAL")

print("\nPlots saved in:")
print(" ", OUTPUT_DIR)

print("\nPlot files:")
print("  A_vs_KAL_ESM_PCA.png")
print("  A_vs_KAL_ESM_UMAP.png")
print("  A_vs_KAL_ESM_tSNE.png")

print("\nCoordinate file:")
print("  A_vs_KAL_ESM_all_embedding_coordinates.xlsx")

Loaded embedding file: testset_A_Ak_CAL_embeddings.npy
Embedding shape: (388, 1280)
Loaded metadata file: testset_A_Ak_CAL_table.xlsx
Metadata rows: 388
Metadata columns: ['id', 'sequence', 'domain_label']
Using label column: domain_label

Normalized label counts:
normalized_label
A      169
CAL    135
Ak      84
Name: count, dtype: int64

Binary class counts:
binary_label
KAL    219
A      169
Name: count, dtype: int64

A count: 169
KAL count: 219
Total: 388

Running PCA
Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_PCA.png
Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_PCA_coordinates.xlsx

Running UMAP


2026-07-14 16:28:00.967194: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-14 16:28:01.147740: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-14 16:28:02.140500: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_UMAP.png
Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_UMAP_coordinates.xlsx

Running t-SNE
t-SNE input dimensions: 50
t-SNE perplexity: 30
Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_tSNE.png
Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_tSNE_coordinates.xlsx
Saved: A_vs_KAL_embedding_plots/A_vs_KAL_ESM_all_embedding_coordinates.xlsx

Finished

Binary classes:
  A: 169
  KAL: 219
  KAL = Ak + CAL

Plots saved in:
  A_vs_KAL_embedding_plots

Plot files:
  A_vs_KAL_ESM_PCA.png
  A_vs_KAL_ESM_UMAP.png
  A_vs_KAL_ESM_tSNE.png

Coordinate file:
  A_vs_KAL_ESM_all_embedding_coordinates.xlsx
